In [ ]:
%pip install -q wandb huggingface_hub
%pip install -q datasets
%pip install -q transformers

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import wandb
from kaggle_secrets import UserSecretsClient
from transformers import get_cosine_schedule_with_warmup
from huggingface_hub import HfApi
from datasets import load_dataset
import torch
import numpy as np
import random
import json

import math
import time
from contextlib import nullcontext
from pathlib import Path

import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from tqdm.auto import tqdm


In [3]:
secrets = UserSecretsClient()

wandb.login(key=secrets.get_secret("wandb"))

hf_api = HfApi(token=secrets.get_secret("huggingface-cli"))
print("HF account:", hf_api.whoami()["name"])

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: zvil (mrzvil) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


HF account: MrZVIL


In [4]:
os.environ['HF_HOME'] = '/kaggle/working/hf_hub'

In [ ]:
experiment_name = "scheduler_small_prenorm_rmsnorm"

run = wandb.init(
    project="hw2-code-generation",
    name=experiment_name,
    config={"purpose": "Evaluate a custom PreNorm Transformer with RMSNorm against the previous PreNorm LayerNorm baseline, keeping model dimensions, data settings, and learning rate schedule fixed. Small version to compare with other models."},
)

In [6]:
def set_global_seed(seed: int) -> None:
    """Set global seed for reproducibility.
    :param int seed: Seed to be set
    """
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
SEED = 67
set_global_seed(SEED)

In [7]:
hf_user_name = hf_api.whoami()["name"]
repo_id = f"{hf_user_name}/hw2-cg-{experiment_name}"

hf_api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    private=True,
    exist_ok=True,
)

RepoUrl('https://huggingface.co/MrZVIL/hw2-cg-baseline_transformer', endpoint='https://huggingface.co', repo_type='model', repo_id='MrZVIL/hw2-cg-baseline_transformer')

In [8]:
dataset = load_dataset("SlavaYus/tinysms_qa_12m")
print(dataset)

Resolving data files:   0%|          | 0/60 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/60 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'code'],
        num_rows: 11838003
    })
    test: Dataset({
        features: ['question', 'code'],
        num_rows: 1919
    })
})


In [9]:
split = dataset["train"].train_test_split(
    test_size=0.10,
    shuffle=True,
    seed=SEED
)

train_dataset = split["train"]
valid_dataset = split["test"]
test_dataset = dataset["test"]

print("Train:", len(train_dataset))
print("Valid:", len(valid_dataset))
print("Test:", len(test_dataset))

Train: 10654202
Valid: 1183801
Test: 1919


In [10]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", device)


Device: cuda


In [ ]:
config = {
    "seed": 67,
    "train_size": 500_000,
    "valid_size": 2_000,
    "tokenizer_train_size": 20_000,
    "vocab_size": 8_000,
    "max_src_len": 256,
    "max_tgt_len": 256,
    "d_model": 128,
    "nhead": 4,
    "num_encoder_layers": 2,
    "num_decoder_layers": 4,
    "dim_feedforward": 512,
    "dropout": 0.1,
    "batch_size": 64,
    "epochs": 3,
    "learning_rate": 5e-4,
    "weight_decay": 0.01,
    "grad_clip": 1.0,
    "log_every": 50,
    "valid_every": 500,
    "scheduler": "cosine",
    "warmup_ratio": 0.05,
    "norm_first": True,
    'norm_eps': 1e-5,
}

In [12]:
train_raw = train_dataset.select(
    range(min(config["train_size"], len(train_dataset)))
)
valid_raw = valid_dataset.select(
    range(min(config["valid_size"], len(valid_dataset)))
)

In [13]:
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=config["vocab_size"],
    min_frequency=2,
    special_tokens=["[PAD]", "[BOS]", "[EOS]", "[UNK]"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
)

tokenizer_data = train_raw.select(
    range(min(config["tokenizer_train_size"], len(train_raw)))
)

def tokenizer_iterator():
    for batch in tokenizer_data.iter(batch_size=1000):
        yield batch["question"] + batch["code"]

tokenizer.train_from_iterator(tokenizer_iterator(), trainer=trainer)

PAD_ID = tokenizer.token_to_id("[PAD]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")

config["vocab_size"] = tokenizer.get_vocab_size()

example = train_raw[0]["code"]
assert tokenizer.decode(tokenizer.encode(example).ids) == example
print("Vocabulary:", config["vocab_size"])




Vocabulary: 16000


In [14]:
def tokenize_batch(batch):
    questions = tokenizer.encode_batch(batch["question"])
    codes = tokenizer.encode_batch(batch["code"])

    return {
        "src_ids": [
            x.ids[:config["max_src_len"] - 1] + [EOS_ID]
            for x in questions
        ],
        "tgt_ids": [
            [BOS_ID] + x.ids[:config["max_tgt_len"] - 2] + [EOS_ID]
            for x in codes
        ],
        "src_truncated": [
            len(x.ids) > config["max_src_len"] - 1 for x in questions
        ],
        "tgt_truncated": [
            len(x.ids) > config["max_tgt_len"] - 2 for x in codes
        ],
    }

train_encoded = train_raw.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_raw.column_names,
    desc="Tokenize train",
)
valid_encoded = valid_raw.map(
    tokenize_batch,
    batched=True,
    remove_columns=valid_raw.column_names,
    desc="Tokenize valid",
)

data_stats = {
    "data/train_size": len(train_raw),
    "data/valid_size": len(valid_raw),
}

for name, data in [("train", train_encoded), ("valid", valid_encoded)]:
    for field in ["src_truncated", "tgt_truncated"]:
        data_stats[f"data/{name}_{field}"] = float(np.mean(data[field]))

print(data_stats)

def collate_fn(examples):
    src = pad_sequence(
        [torch.tensor(x["src_ids"], dtype=torch.long) for x in examples],
        batch_first=True,
        padding_value=PAD_ID,
    )
    tgt = pad_sequence(
        [torch.tensor(x["tgt_ids"], dtype=torch.long) for x in examples],
        batch_first=True,
        padding_value=PAD_ID,
    )
    return src, tgt

loader_generator = torch.Generator().manual_seed(config["seed"])

train_loader = DataLoader(
    train_encoded,
    batch_size=config["batch_size"],
    shuffle=True,
    generator=loader_generator,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=device.type == "cuda",
)
valid_loader = DataLoader(
    valid_encoded,
    batch_size=config["batch_size"],
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=device.type == "cuda",
)

Tokenize train:   0%|          | 0/500000 [00:00<?, ? examples/s]

Tokenize valid:   0%|          | 0/20000 [00:00<?, ? examples/s]

{'data/train_size': 500000, 'data/valid_size': 20000, 'data/train_src_truncated': 2e-06, 'data/train_tgt_truncated': 0.003262, 'data/valid_src_truncated': 0.0, 'data/valid_tgt_truncated': 0.00295}


In [ ]:
class BasicFeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.linear1 = nn.Linear(
            config['d_model'],
            config['dim_feedforward'],
        )
        self.linear2 = nn.Linear(
            config['dim_feedforward'],
            config['d_model'],
        )
        self.dropout = nn.Dropout(config['dropout'])
    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

FFN = BasicFeedForward

class EncoderLayer(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.self_attn = nn.MultiheadAttention(
            embed_dim=config['d_model'],
            num_heads=config['nhead'],
            dropout=config['dropout'],
            batch_first=True,
        )

        self.ffn = FFN(config)
        self.norm1 = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )
        self.norm2 = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )
        self.dropout1 = nn.Dropout(config['dropout'])
        self.dropout2 = nn.Dropout(config['dropout'])

    def attention(self, x, padding_mask):
        output, _ = self.self_attn(
            query=x,
            key=x,
            value=x,
            key_padding_mask=padding_mask,
            need_weights=False
        )
        return output

    def forward(self, x, padding_mask=None):
        x = x + self.dropout1(
            self.attention(self.norm1(x), padding_mask)
        )
        x = x + self.dropout2(self.ffn(self.norm2(x)))

        return x

class DecoderLayer(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=config['d_model'],
            num_heads=config['nhead'],
            dropout=config['dropout'],
            batch_first=True,
        )

        self.self_attn = nn.MultiheadAttention(
            embed_dim=config['d_model'],
            num_heads=config['nhead'],
            dropout=config['dropout'],
            batch_first=True,
        )

        self.ffn = FFN(config)

        self.norm1 = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )
        self.norm2 = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )
        self.norm3 = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )

        self.dropout1 = nn.Dropout(config["dropout"])
        self.dropout2 = nn.Dropout(config["dropout"])
        self.dropout3 = nn.Dropout(config["dropout"])

    def self_attention(self, x, causal_mask, padding_mask):
        output, _ = self.self_attn(
            query=x,
            key=x,
            value=x,
            attn_mask=causal_mask,
            key_padding_mask=padding_mask,
            need_weights=False,
        )
        return output

    def cross_attention(self, x, memory, memory_padding_mask):
        output, _ = self.cross_attn(
            query=x,
            key=memory,
            value=memory,
            key_padding_mask=memory_padding_mask,
            need_weights=False,
        )
        return output

    def forward(self, x, memory, causal_mask, padding_mask=None, memory_padding_mask=None):
        x = x + self.dropout1(self.self_attention(self.norm1(x), causal_mask, padding_mask))
        x = x + self.dropout2(self.cross_attention(self.norm2(x), memory, memory_padding_mask))
        x = x + self.dropout3(self.ffn(self.norm3(x)))

        return x

In [ ]:
class CodeTransformer(nn.Module):
    def __init__(self, config, pad_id):
        super().__init__()

        self.pad_id = pad_id
        self.d_model = config["d_model"]

        self.token_embedding = nn.Embedding(
            config["vocab_size"],
            self.d_model,
            padding_idx=pad_id,
        )
        self.position_embedding = nn.Embedding(
            max(config["max_src_len"], config["max_tgt_len"]),
            self.d_model,
        )
        self.embedding_dropout = nn.Dropout(config["dropout"])

        self.encoder_layers = nn.ModuleList([
            EncoderLayer(config)
            for _ in range(config["num_encoder_layers"])
        ])
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(config)
            for _ in range(config["num_decoder_layers"])
        ])

        self.encoder_norm = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )
        self.decoder_norm = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )

        self.output = nn.Linear(
            self.d_model,
            config["vocab_size"],
        )

        self.reset_parameters()

    def reset_parameters(self):
        for layers in (self.encoder_layers, self.decoder_layers):
            for parameter in layers.parameters():
                if parameter.dim() > 1:
                    nn.init.xavier_uniform_(parameter)

        nn.init.normal_(self.token_embedding.weight, std=0.02)
        nn.init.normal_(self.position_embedding.weight, std=0.02)

        with torch.no_grad():
            self.token_embedding.weight[self.pad_id].zero_()

    def embed(self, ids):
        positions = torch.arange(ids.size(1), device=ids.device)

        x = (
            self.token_embedding(ids) * math.sqrt(self.d_model)
            + self.position_embedding(positions)
        )
        return self.embedding_dropout(x)

    def encode(self, src):
        padding_mask = src.eq(self.pad_id)
        memory = self.embed(src)

        for layer in self.encoder_layers:
            memory = layer(memory, padding_mask=padding_mask)

        return self.encoder_norm(memory)

    def decode(self, tgt_input, memory, memory_padding_mask=None):
        length = tgt_input.size(1)

        causal_mask = torch.triu(
            torch.ones(
                length,
                length,
                dtype=torch.bool,
                device=tgt_input.device,
            ),
            diagonal=1,
        )

        padding_mask = tgt_input.eq(self.pad_id)
        x = self.embed(tgt_input)

        for layer in self.decoder_layers:
            x = layer(
                x,
                memory,
                causal_mask=causal_mask,
                padding_mask=padding_mask,
                memory_padding_mask=memory_padding_mask,
            )

        return self.output(self.decoder_norm(x))

    def forward(self, src, tgt_input):
        memory = self.encode(src)

        return self.decode(
            tgt_input,
            memory,
            memory_padding_mask=src.eq(self.pad_id),
        )

In [16]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    loss_sum = 0.0
    correct = 0
    token_count = 0

    for src, tgt in loader:
        src = src.to(device, non_blocking=True)
        tgt = tgt.to(device, non_blocking=True)
        labels = tgt[:, 1:]

        with amp_context():
            logits = model(src, tgt[:, :-1])
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                ignore_index=PAD_ID,
                reduction="sum",
            )

        mask = labels.ne(PAD_ID)
        loss_sum += loss.item()
        correct += ((logits.argmax(-1) == labels) & mask).sum().item()
        token_count += mask.sum().item()

    return {
        "val/loss": loss_sum / token_count,
        "val/token_accuracy": correct / token_count,
    }

In [ ]:
use_amp = device.type == "cuda"

def amp_context():
    return (
        torch.autocast(device_type="cuda", dtype=torch.float16)
        if use_amp else nullcontext()
    )

def train(
    model,
    train_loader,
    valid_loader,
    optimizer,
    config,
    device,
    pad_id,
    run,
    output_dir,
    scheduler=None,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    global_step = 0
    best_val_loss = float("inf")
    history = []

    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    for epoch in range(1, config["epochs"] + 1):
        model.train()

        epoch_loss_sum = 0.0
        epoch_tokens = 0
        window_loss_sum = 0.0
        window_tokens = 0
        window_start = time.perf_counter()

        progress = tqdm(train_loader, desc=f"Epoch {epoch}")

        for batch_idx, (src, tgt) in enumerate(progress, start=1):
            src = src.to(device, non_blocking=True)
            tgt = tgt.to(device, non_blocking=True)

            tgt_input = tgt[:, :-1]
            labels = tgt[:, 1:]
            n_tokens = labels.ne(pad_id).sum().item()

            optimizer.zero_grad(set_to_none=True)

            with amp_context():
                logits = model(src, tgt_input)
                loss = F.cross_entropy(
                    logits.reshape(-1, logits.size(-1)),
                    labels.reshape(-1),
                    ignore_index=pad_id,
                )

            if not torch.isfinite(loss).item():
                raise RuntimeError("Loss became NaN or Inf")

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)
            grad_norm = nn.utils.clip_grad_norm_(
                model.parameters(),
                config["grad_clip"],
            )

            lr_used = optimizer.param_groups[0]["lr"]

            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()

            step_skipped = scaler.get_scale() < old_scale

            if not step_skipped:
                global_step += 1

                if scheduler is not None:
                    scheduler.step()

            batch_loss_sum = loss.item() * n_tokens
            epoch_loss_sum += batch_loss_sum
            epoch_tokens += n_tokens
            window_loss_sum += batch_loss_sum
            window_tokens += n_tokens

            should_log = (
                batch_idx % config["log_every"] == 0
                or batch_idx == len(train_loader)
            )

            should_valid = (
                batch_idx % config["valid_every"] == 0
                or batch_idx == len(train_loader)
            )

            if should_log:
                elapsed = time.perf_counter() - window_start
                window_loss = window_loss_sum / window_tokens

                run.log({
                    "global_step": global_step,
                    "epoch": epoch,
                    "train/loss": window_loss,
                    "train/lr": lr_used,
                    "train/grad_norm": grad_norm.item(),
                    "train/target_tokens_per_second": (
                        window_tokens / elapsed
                    ),
                })
                progress.set_postfix(loss=window_loss)

                window_loss_sum = 0.0
                window_tokens = 0
                window_start = time.perf_counter()
            if should_valid:
                metrics = evaluate(model, valid_loader)
                model.train()
                run.log(metrics)
                if metrics["val/loss"] < best_val_loss:
                    best_val_loss = metrics["val/loss"]
        
                    torch.save({
                        "model_state_dict": model.state_dict(),
                        "config": dict(config),
                        "pad_id": pad_id,
                        "epoch": epoch,
                        "global_step": global_step,
                        "val_loss": best_val_loss,
                        "wandb_run_id": run.id,
                    }, output_dir / "best.pt")
        
                    run.summary["best/val_loss"] = best_val_loss
                    run.summary["best/epoch"] = epoch
        metrics = evaluate(model, valid_loader)
        model.train()
        metrics.update({
            "global_step": global_step,
            "epoch": epoch,
            "train/epoch_loss": epoch_loss_sum / epoch_tokens,
        })

        run.log(metrics)
        history.append(metrics)
        print(f"Epoch {epoch}: {metrics}")

        if metrics["val/loss"] < best_val_loss:
            best_val_loss = metrics["val/loss"]

            torch.save({
                "model_state_dict": model.state_dict(),
                "config": dict(config),
                "pad_id": pad_id,
                "epoch": epoch,
                "global_step": global_step,
                "val_loss": best_val_loss,
                "wandb_run_id": run.id,
            }, output_dir / "best.pt")

            run.summary["best/val_loss"] = best_val_loss
            run.summary["best/epoch"] = epoch

    return history

In [ ]:
set_global_seed(config["seed"])

model = CodeTransformer(config, PAD_ID).to(device)
model.compile()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)

num_training_steps = len(train_loader) * config["epochs"]
num_warmup_steps = int(num_training_steps * config["warmup_ratio"])

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

config.update({
    "num_training_steps": num_training_steps,
    "num_warmup_steps": num_warmup_steps,
})
run.config.update(config)

run.summary.update(data_stats)
run.summary["model/parameters"] = sum(p.numel() for p in model.parameters())
run.summary["device"] = str(device)
run.summary["torch_version"] = str(torch.__version__)

output_dir = Path("/kaggle/working/baseline") / run.id
output_dir.mkdir(parents=True, exist_ok=True)
tokenizer.save(str(output_dir / "tokenizer.json"))

print("Parameters:", run.summary["model/parameters"])

Parameters: 50179712


In [ ]:
train(
    model,
    train_loader,
    valid_loader,
    optimizer,
    config,
    device,
    PAD_ID,
    run,
    output_dir,
    scheduler=scheduler
)

Epoch 1:   0%|          | 0/15625 [00:00<?, ?it/s]

W1013 10:40:22.191000 1204 torch/_dynamo/variables/tensor.py:1379] [0/0] Graph break from `Tensor.item()`, consider setting:
W1013 10:40:22.191000 1204 torch/_dynamo/variables/tensor.py:1379] [0/0]     torch._dynamo.config.capture_scalar_outputs = True
W1013 10:40:22.191000 1204 torch/_dynamo/variables/tensor.py:1379] [0/0] or:
W1013 10:40:22.191000 1204 torch/_dynamo/variables/tensor.py:1379] [0/0]     env TORCHDYNAMO_CAPTURE_SCALAR_OUTPUTS=1
W1013 10:40:22.191000 1204 torch/_dynamo/variables/tensor.py:1379] [0/0] to include these operations in the captured graph.
W1013 10:40:22.191000 1204 torch/_dynamo/variables/tensor.py:1379] [0/0] 
W1013 10:40:22.191000 1204 torch/_dynamo/variables/tensor.py:1379] [0/0] Graph break: from user code at:
W1013 10:40:22.191000 1204 torch/_dynamo/variables/tensor.py:1379] [0/0]   File "/tmp/ipykernel_1204/3936584717.py", line 50, in forward
W1013 10:40:22.191000 1204 torch/_dynamo/variables/tensor.py:1379] [0/0]     hidden = self.transformer(
W1013 10

Epoch 1: {'val/loss': 0.6744380075596893, 'val/token_accuracy': 0.8371618661606852, 'global_step': 15622, 'epoch': 1, 'train/epoch_loss': 1.1189774991942043}


Epoch 2:   0%|          | 0/15625 [00:00<?, ?it/s]

: 

In [ ]:
@torch.no_grad()
def generate(model, question, max_new_tokens=128):
    model.eval()

    src_ids = tokenizer.encode(question).ids[:config["max_src_len"] - 1]
    src = torch.tensor([src_ids + [EOS_ID]], device=device)
    src_padding = src.eq(PAD_ID)

    memory = model.encode(src)
    generated = torch.tensor([[BOS_ID]], device=device)

    for _ in range(min(max_new_tokens, config["max_tgt_len"] - 1)):
        length = generated.size(1)
        causal_mask = torch.triu(
            torch.ones(length, length, device=device, dtype=torch.bool),
            diagonal=1,
        )
        logits = model.decode(
            generated,
            memory,
            memory_padding_mask=src_padding,
        )[:, -1]
        logits[:, [PAD_ID, BOS_ID]] = -float("inf")
        next_token = logits.argmax(-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=1)

        if next_token.item() == EOS_ID:
            break

    return tokenizer.decode(generated[0].tolist(), skip_special_tokens=True)


checkpoint = torch.load(
    output_dir / "best.pt", map_location=device, weights_only=True
)
model.load_state_dict(checkpoint["model_state_dict"])

table = wandb.Table(columns=["question", "reference", "prediction"])

for example in valid_raw.select(range(min(5, len(valid_raw)))):
    prediction = generate(model, example["question"])
    table.add_data(example["question"], example["code"], prediction)
    print("Question:", example["question"])
    print("Prediction:\n", prediction, "\n")

run.log({"val/examples": table})
run.finish()

Question: Emily and her friend Sarah decided to start a lemonade stand. Emily made 20 cups of lemonade, which is 3/4 of the total amount of lemonade they have. Sarah made twice as many cups of lemonade as Emily and an additional 10 cups of orange juice. How much liquid in total do they have?
Prediction:
 def solve() -> int:
    # Step 1: Calculate the total number of lemonade in the total number of lemonade in the total number of lemonade in the question
    # Step 2: Calculate the number of lemonade in the total number of lemonade in the total number of lemonade in the total number of lemonade in the total number of lemonade in the total number of lemonade in the total
    lemonade_cups = 10
    lemonade_cups = lemonade_cups * 2/4
    lemonade_cups = lemonade_cups * 10
    lemonade_cups = lemonade_cups * 10
    total_cups = lemonade_cups + lemonade_cups
    result = 

Question: In a garden, there are 2000 flowers. Every day, 300 flowers bloom, while 500 flowers wither away. How many f

epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▅▅▅▅▅▅▅▅██████████████
global_step,▁▁▁▁▂▂▂▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇█████
train/epoch_loss,█▃▁
train/grad_norm,▃▄▁▄▃▃▃▃▃▃▅▄▃▃▄▅▄▃▃▆▇▅▄▆▄▅▄▅▅▄▅▆▅▅▃▄▅▄▅█
train/loss,█▅▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/target_tokens_per_second,▁▆█▆▂▅▅▇▅▅▅▆▅▄▄▃▅▄▄▅▆▄▄▄▄▄▃▅▅▃▄▄▂▆▅▅▃▅▄▃
val/loss,█▃▁
val/token_accuracy,▁▆█
best/epoch,3
best/val_loss,1.15746


In [ ]:
hf_api.upload_folder(
    repo_id=repo_id,
    repo_type="model",
    folder_path=str(output_dir),
    path_in_repo=f"runs/{run.id}",
)

CommitInfo(commit_url='https://huggingface.co/MrZVIL/hw2-cg-baseline_transformer/commit/7e71a4df79099aec224e46aa404529e8e7033d55', commit_message='Upload folder using huggingface_hub', commit_description='', oid='7e71a4df79099aec224e46aa404529e8e7033d55', pr_url=None, repo_url=RepoUrl('https://huggingface.co/MrZVIL/hw2-cg-baseline_transformer', endpoint='https://huggingface.co', repo_type='model', repo_id='MrZVIL/hw2-cg-baseline_transformer'), pr_revision=None, pr_num=None)

In [ ]:
@torch.inference_mode()
def generate_batch(
    model,
    tokenizer,
    questions,
    max_src_len,
    max_new_tokens=64,
):
    if not questions:
        return []

    device = next(model.parameters()).device
    model.eval()

    max_positions = model.position_embedding.num_embeddings
    if not 1 <= max_new_tokens <= max_positions:
        raise ValueError("max_new_tokens exceeds positional embedding capacity")
    if not 1 <= max_src_len <= max_positions:
        raise ValueError("Invalid max_src_len")

    encoded = tokenizer.encode_batch(questions)
    src = pad_sequence(
        [
            torch.tensor(
                item.ids[:max_src_len - 1] + [EOS_ID],
                dtype=torch.long,
            )
            for item in encoded
        ],
        batch_first=True,
        padding_value=PAD_ID,
    ).to(device)

    src_padding = src.eq(PAD_ID)
    memory = model.encode(src)

    generated = torch.full(
        (len(questions), 1),
        BOS_ID,
        dtype=torch.long,
        device=device,
    )
    finished = torch.zeros(
        len(questions), dtype=torch.bool, device=device
    )

    causal_mask = torch.triu(
        torch.ones(
            max_new_tokens,
            max_new_tokens,
            dtype=torch.bool,
            device=device,
        ),
        diagonal=1,
    )

    forbidden_ids = [
        tokenizer.token_to_id(token)
        for token in ["[PAD]", "[BOS]", "[UNK]"]
        if tokenizer.token_to_id(token) is not None
    ]

    for _ in range(max_new_tokens):
        length = generated.size(1)

        logits = model.decode(
            generated,
            memory,
            memory_padding_mask=src_padding,
        )[:, -1]
        logits[:, forbidden_ids] = -float("inf")
        next_tokens = logits.argmax(dim=-1)

        next_tokens = next_tokens.masked_fill(finished, EOS_ID)
        generated = torch.cat(
            [generated, next_tokens.unsqueeze(1)],
            dim=1,
        )

        finished |= next_tokens.eq(EOS_ID)
        if finished.all().item():
            break

    answers = []
    for ids in generated[:, 1:].cpu().tolist():
        if EOS_ID in ids:
            ids = ids[:ids.index(EOS_ID)]
        answers.append(tokenizer.decode(ids, skip_special_tokens=True))

    return answers

In [ ]:
def test_collate_fn(examples):
    return {"question": [example["question"] for example in examples]}


test_loader = DataLoader(
    test_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
    collate_fn=test_collate_fn,
    num_workers=0,
)

In [ ]:
with open("/kaggle/working/submission_batch.jsonl", "w", encoding="utf-8") as f:
    for tasks in tqdm(test_loader, desc="Generating"):
        with amp_context():
            codes = generate_batch(
                model=model,
                tokenizer=tokenizer,
                questions=tasks["question"],
                max_src_len=config["max_src_len"],
                max_new_tokens=64,
            )
        for code in codes:
            f.write(json.dumps({"code": code}, ensure_ascii=False) + "\n")

Generating:   0%|          | 0/60 [00:00<?, ?it/s]